# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)

This notebook simulates streaming ingestion by generating synthetic FHIR observations/patients and appending them to the Bronze Iceberg table.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, expr, lit

# Setup local Iceberg warehouse path (reusing from Day 1)
warehouse_path = "/tmp/iceberg_warehouse"
os.makedirs(warehouse_path, exist_ok=True)

spark = SparkSession.builder \
    .appName("Day02_Streaming_Ingestion") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
import json
import uuid as pyuuid
import time
import pandas as pd

# Ensure the table exists (simulate Day 1 setup if run standalone)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.bronze")
spark.sql("""
CREATE TABLE IF NOT EXISTS local.bronze.raw_fhir_payload (
    payload_id STRING,
    source_system_id STRING,
    raw_payload_json STRING,
    ingestion_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

# Generate Synthetic FHIR data
def generate_fhir_patient():
    return json.dumps({
        "resourceType": "Patient",
        "id": str(pyuuid.uuid4()),
        "name": [{"family": "Doe", "given": ["John"]}]
    })

synthetic_data = [generate_fhir_patient() for _ in range(5)]

# Simulate reading a stream by creating a streaming DataFrame from a rate source
rate_stream = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

# Create synthetic stream processing logic
streaming_df = rate_stream \
    .withColumn("payload_id", expr("uuid()")) \
    .withColumn("source_system_id", lit("simulated_fhir_stream")) \
    .withColumn("raw_payload_json", lit(synthetic_data[0])) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("payload_id", "source_system_id", "raw_payload_json", "ingestion_timestamp")

# Write stream to Iceberg Table
checkpoint_path = "/tmp/iceberg_checkpoints/fhir_stream"
query = streaming_df.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .toTable("local.bronze.raw_fhir_payload")

print("Started Streaming into Iceberg. Waiting 10 seconds...")
time.sleep(10)
query.stop()
query.awaitTermination()
print("Streaming stopped.")

In [ ]:
# Verify Data was written to Bronze
spark.sql("SELECT * FROM local.bronze.raw_fhir_payload").show(truncate=False)
print(f"Total records in Bronze: {spark.sql('SELECT count(*) FROM local.bronze.raw_fhir_payload').collect()[0][0]}")